# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/13aakash/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
%pip -q install duckdb huggingface_hub

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste HF token: ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
fact_daily = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
dim_content = f"read_parquet('{REL}/dim_content.parquet')"
dim_clients = f"read_parquet('{REL}/dim_clients.parquet')"

print(con.sql(f"SELECT COUNT(*) FROM {fact_daily}").fetchone())

(9841378,)


## 1. Unit of analysis + time window

Unit of analysis: One row = one content item (content_hash_id), aggregated over a fixed 30-day window from fact_content_daily_performance.
Time window: report_date in March 2026 (month=2026-03) — a mid-panel month, never _sample (which is the sealed final month).

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Grain check: one row per (report_date, client_hash_id, content_hash_id) — should be EMPTY
con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {fact_daily}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING c > 1
    LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,c


## 2. Fields: feature / label / context / excluded

Feature: gsc_impressions, gsc_clicks, gsc_avg_position — all daily, all knowable before the decision point.
Label/proxy (for the leakage demo only): declined_within_month — derived from the same impressions this notebook uses, so it's a within-window teaching proxy, not a real forward label.
Context: client_hash_id, content_hash_id — grouping/joining only, never features.
Excluded: GA4 columns wherever ga4_data_available IS NOT TRUE (zero there means "not tracked," not "no engagement"); any product decision flags (none exist in this release, but noting the rule).

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Confirm which columns exist so the exclusion list above is accurate
con.sql(f"DESCRIBE SELECT * FROM {fact_daily}").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 3. Verify it with queries (grain, counts, missing values, windows)

Three queries below: grain (repeated from §1 for completeness), row count + date span, and availability filtered with IS TRUE.

In [5]:
#@title Code cell A — row count + date span
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
q2 = con.sql(f"""
    SELECT COUNT(*) AS row_count,
           MIN(report_date) AS min_date, MAX(report_date) AS max_date,
           COUNT(DISTINCT content_hash_id) AS n_content,
           COUNT(DISTINCT client_hash_id) AS n_clients
    FROM {fact_daily}
""").df()
q2

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,min_date,max_date,n_content,n_clients
0,9841378,2026-03-01,2026-03-31,331437,55


In [6]:
#@title Code cell B — availability with IS TRUE
q3 = con.sql(f"""
    SELECT
      COUNT(*) AS total_rows,
      SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available,
      SUM(CASE WHEN ga4_data_available IS NOT TRUE THEN 1 ELSE 0 END) AS ga4_unavailable_or_null
    FROM {fact_daily}
""").df()
q3

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available,ga4_unavailable_or_null
0,9841378,413966.0,9427412.0


413966 of 9841378 rows have GA4 available

In [7]:
#@title Code cell C — build the 5-feature frame
features = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS impressions_30d,
        AVG(gsc_avg_position) AS avg_position_30d,
        SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS ctr_30d,
        COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS days_with_impressions_30d,
        SUM(CASE WHEN report_date >= DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS second_half_impr,
        SUM(CASE WHEN report_date <  DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS first_half_impr
    FROM {fact_daily}
    GROUP BY content_hash_id
    HAVING first_half_impr >= 50
""").df()

features['momentum_ratio'] = features['second_half_impr'] / features['first_half_impr']
print(f"{len(features):,} content items with enough volume")
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

92,548 content items with enough volume


,content_hash_id,impressions_30d,avg_position_30d,ctr_30d,days_with_impressions_30d,second_half_impr,first_half_impr,momentum_ratio
0,content_7a105f548d9c6916,6523.0,7.209549,0.001073,31,2350.0,4173.0,0.563144
1,content_a3ea9792f793ec72,453.0,2.987198,0.000000,31,208.0,245.0,0.848980
2,content_36c36abc7650d7af,5630.0,6.724039,0.001066,31,1925.0,3705.0,0.519568
3,content_a7da352b73b02668,4944.0,7.244844,0.002629,31,2504.0,2440.0,1.026230
4,content_1855a661b4d36130,429.0,4.209227,0.002331,31,189.0,240.0,0.787500


impressions_30d — available: trailing sum of already-observed daily search data.

avg_position_30d — available: daily position values are known before month-end.

ctr_30d — available: computed purely from trailing clicks/impressions.

days_with_impressions_30d — available: counts only days already elapsed.

momentum_ratio — caveat, not a clean model feature: both halves fall inside the window, so it's technically knowable, but as the leakage cell shows, it's too close to the label to trust as a feature here — kept for demonstration only.

In [8]:
#@title Code cell D — the leakage trap (fixed)
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

features['declined_within_month'] = (
    features['second_half_impr'] < 0.8 * features['first_half_impr']
).astype(int)

print("Base rate (declined):", features['declined_within_month'].mean())

features['leaky_decline_signal'] = (
    (features['second_half_impr'] - features['first_half_impr']) / features['first_half_impr']
)

y = features['declined_within_month']

# LEAKY: includes the direct give-away
X_leaky = features[['impressions_30d','avg_position_30d','ctr_30d',
                     'days_with_impressions_30d','leaky_decline_signal']].fillna(0)
Xtr, Xte, ytr, yte = train_test_split(X_leaky, y, test_size=0.3, random_state=42, stratify=y)
leaky_model = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
print("WITH leaky feature, accuracy:", leaky_model.score(Xte, yte))

# HONEST: momentum_ratio removed too — it's the same leak in disguise
X_honest = features[['impressions_30d','avg_position_30d','ctr_30d',
                      'days_with_impressions_30d']].fillna(0)
Xtr, Xte, ytr, yte = train_test_split(X_honest, y, test_size=0.3, random_state=42, stratify=y)
honest_model = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
print("WITHOUT leaky feature, accuracy:", honest_model.score(Xte, yte))

#momentum_ratio was also dropped from the honest set because it's mathematically derived from the same halves as the label — a second instance of the same leak.

Base rate (declined): 0.28643514716687557
WITH leaky feature, accuracy: 0.9996398343237889
WITHOUT leaky feature, accuracy: 0.7228885287232126


## 4. Data limits

This March slice only reflects clients whose gsc_data_start predates March 2026 — clients onboarded mid-month are under-represented. GA4-derived signals are unreliable wherever ga4_data_available IS NOT TRUE — from §3's query, 413,966 of 9,841,378 rows (about 4.2%) have GA4 available, leaving 9,427,412 rows where GA4 columns should not be trusted. momentum_ratio is a within-month proxy, not a validated forward-looking label — a real capstone label needs a separate future window (April), which this notebook doesn't build.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# How many clients in this slice started tracking before March 2026?
con.sql(f"""
    SELECT COUNT(*) AS clients_with_early_start
    FROM {dim_clients}
    WHERE gsc_data_start < DATE '2026-03-01'
""").df()

,clients_with_early_start
0,52


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.